<a href="https://colab.research.google.com/github/snr-laboratory/snrlab-ic-q-pix-v1/blob/main/dev_journals/kgosine_202507_a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Buffering Between Integrator and Comparator

With the ourput of the integrator being directly connected to the input of the comparator, the clock in the comparator causes a voltage kickback at the integrator output. With no buffer between them, the amplitude of that kickback is 24mV

<img src="https://drive.google.com/uc?id=1Ib6fsT93qBGWp6iDlszJH4OOj-70LMWa"  height="500" >


### NMOS Source Follower Buffer

There is a voltage drop at the output (despite the buffer being inside the opamp feedback loop) from 900mV to about 880mV.  

<img src="https://drive.google.com/uc?id=1HHZkm8jLcjTtUvHOsZVt-g6ZQ9cy5Q1G"  height="400" >

It did however reduce the kickback at the integrator output to 13mV

<img src="https://drive.google.com/uc?id=1o4TzT7O6rhhsiDEodgivYf5rSqej6iFM"  height="400" >

If we treat the buffer as being outside of the feedback of the opamp we see the integrator output is protected while the buffer output node (connected to the comparator) still has the kickback from the comparator.

The kickback on the buffered output is 6.5mV and on the integrator output is only 0.85mV.

<img src="https://drive.google.com/uc?id=16x4FUXs0CDX5q20-A1cOdn6hCL6HHqfx"  height="400" >
<img src="https://drive.google.com/uc?id=1TOL7LCJ2hHus7zclSHvmsu_-zGzXQt7d"  height="400" >




The problem with this set up is that the buffered output has been reduced by Vgs of the nmos source-follower and therefore begins at 345mV. This however is too low for the comparator to work properly. Even if the Vref_comp is set to a value slightly above this (350mV), it fails to trigger until the buffered output is at leas 408mV; this indicates the comparator common mode is around 0.4V and therefore can't compare properly with inputs below this voltage.

### NMOS and PMOS Source Followers
To deal with this problem of the Vgs drop of the output, a PMOS source follower can be added which adds an addition Vgs drop in the opposite direction. First, placed within the opamp feedback loop:

<img src="https://drive.google.com/uc?id=1PBtcalHnnzuBU6UMgTD0hiwGhdp_Gukh"  height="400" >
<img src="https://drive.google.com/uc?id=1qHpuBECv44SYyzjRbtyhsN1N03ta9CgI"  height="400" >

The kickback on the buffered output is 15.8mV and the comparator is able to fire when it the output exceeds threshold. It also sits at 900mV before any charge lands on the pixel as expected for the opamp.

We can try placing it outisde of the opamp feedback loop as a separate circuit:

<img src="https://drive.google.com/uc?id=1oOa_qTAlceG_RcwQX4misi0ZIbwhJ13T"  height="400" >
<img src="https://drive.google.com/uc?id=12IM_uQctxpGELVL-Rbw_d4APnweHe7Rk"  height="400" >



The output of the integrator is protected from the kickback of the comparator and has about 0.4mV of kickback. The output of the buffer is shifted from 0.9V to 1.1V due to the mismatch in threshold voltages of the PMOS and NMOS and also experiences 24.5mV of kickback. This is still able to trigger the comparator.

With two separate current sources, this configuration consumes a large amount of power.











### Addition of RC Low-Pass Filter After Source Follower Buffer

The nmos source follower buffer was placed inside the opamp feedback loop to avoid the issue of the voltage drop exceeding the common mode range of the comparator. Addition of an RC low pass filter (R=1k, C=100f, $\tau$= 100ps) is meant to block the high frequency (50MHz clock frequency) kickback at the opamp output. This works well without distorting the response speed of the opamp (opamp output steps in time are the same with and without the LPF).

<img src="https://drive.google.com/uc?id=1mvQStdiLXyCmjmJHyzW1jLEdcSll3BMl"  height="500" >


<img src="https://drive.google.com/uc?id=1S4sapQ52Dm10l6HUzY0Mv3zyFXA5hpZK"  height="500" >

[Blue trace is opamp output with only source-follower, Pink is output of source-follower plus RC low-pass filter]


The kickback on the output of the LPF (which is the input to the comparator, pink trace) is only 3.3mV and on the output of the integrator is only 2.9mV (green trace).

This can be compared to the previous case of simply the nmos source follower (inside the opamp feedback) having an output kickback of 13mV. Note that the buffered output is still reduced to about 884mV (rather than the 900mV) at baseline from the nmos source follower.

<img src="https://drive.google.com/uc?id=1Orq9-V__QKmNDrYVZvqLjmeX8xr3SEPo"  height="500" >



Increasing the current drive of the opamp helps overcome the response time of the opamp output through the RC low-pass filter. To check this, I removed the source-follower buffer and placed only the low pass filter between the original opamp and the comparator.

Removal of the source follower output stage didn't make a difference in the timing response of the opamp but it did increase the size of the kickback at the LPF output from 3.3mV to 4.3mV.

<img src="https://drive.google.com/uc?id=1arys9F0rVpB2bEQV1qYRVqgcnAy12M8w"  height="500" >


### Sweep of Low Pass Filter Capacitor Size

Compared the response time of the LPF_output with and without a source follower buffer inside the opamp loop for various values of the blocking capacitor.

With the source follower at the opamp output:

<img src="https://drive.google.com/uc?id=13ybd9KiFaMOmanoelQN4WOTzWhs9SKOa"  height="500" >

For blocking capacitors of size 100fF-500fF, there is no significant delay in the rise time of the integrator output. The pink trace is the rise time with no low pass filter (capacitance of 0F)

<img src="https://drive.google.com/uc?id=1FtxTGdam8J1Seh1oaiS1BiUsG6I6WAsS"  height="500" >


If we remove the source follower at the output and hook the opamp directly to the low pass filter:

<img src="https://drive.google.com/uc?id=1pBIZ_gWxcTtm2FQXs3QzyJAndQP3rLMh"  height="500" >

We see that there is a delay in the rise time of the integrator as the blocking capacitor gets larger.

Therefore, while the blocking capacitor is responsible for minimizing the kickback voltage, the source follower output stage is necessary so that the capacitor doesn't significantly affect the rise time of the integrator.











### Testing Source Follower Output + LPF in QPix Loop

Vref_comp=910mV:
<img src="https://drive.google.com/uc?id=14C7AchQg5gMRhZgyBT90Vpr7BLyDuLD5"  height="500" >

On the left is the qpix loop without any buffering. On the left is with the source follower output stage and a 500fF blocking capacitor.
There is an anomolous call for replenishment at the beginning of the simulation in the case of no buffer which causes the simulations to be out of sync with each other.
But the buffered output is still an 8mV step (which it should be for a 20ns, 20nA current on pixel pulse) and has a fast rise time (24ns which is the duration of the current landing on the pixel)


<img src="https://drive.google.com/uc?id=1YkAKu4Hzqbe0LWMTRPb_ma_kvnTk4vZj"  height="500" >

## Buffering Between Comparator and Current Injection

At the output of the comparator there is also significant kickback from the clock which causes small (less than 2nA) current injections on clock edges. These small pulses are not symmetric though on clock rising and falling and therefore contribute to leakage current of the replenishment charge. Note that buffering the integrator output (red trace) helped reduce this:

<img src="https://drive.google.com/uc?id=1KfnmFI0ihF4I3JJmA0sEozdEekC0-dDd"  height="500" >

The amount of leakage charge on a clock edge is proportional to the comparator output voltage which is larger (upto 0.5V) before the comparator goes logic high.

<img src="https://drive.google.com/uc?id=1zN9NqQeZptECgbvJlDaEWF2ExXSlk-5P"  height="500" >

This further supports the need for a buffer between the comparator output and the current replenishment block.

## Characterizing Dynamic Comparator

The size of the (capacitive) load on the dynamic compartor affects the fidelity of its outputs, particularly the asymmetry of load capacitance.
Sweeping capacitive loads on both outputs of the comparator, when there is

N_load=100f
P_load= 70fF-110fF the behavior of both outputs is correct
P_load= < 70fF or > 110fF causes both outputs to be incorrect
The greater the difference between load capacitances, the larger the voltage spikes on one of the outputs.

Increasing the output current drive of the comparator would help with the drive strength, unknown if it will help with the incorrect logic outputs from mismatched loads.

Since the comparator output is digital, a passive logic gate like an inverter should work to increase the drive strength and minimize the clock pulses on the voltage. Both outputs after the inverter have a 10fF load.


<img src="https://drive.google.com/uc?id=18yg3fs_rGKqCFf4TxqCA2NtErbeUcOdV"  height="400" >

<img src="https://drive.google.com/uc?id=14vFc6MteOOZV4HlUOGndQK8-qDzoJYgv"  height="500" >

We can see that the inverted outputs have negligible spikes during clock edges compared to the non-inverted outputs.




### Inserting into QPix Loop

The important thing here is to note that the two comparator outputs are not inverses of each other because of the 50% duty cycle of the high output. i.e. when the comparator is not triggered, OutN is high 50% of the time and OutP (the output we want) is always low. Therefore, since the charge replenishment block requires a comparator output (Q) and a comparator output inverse (Qbar), we shouldn't try to use OutN as Qbar.

The best way to buffer the output we want (OutP) was to add two inverters. The first inverter created Qbar (which is somewhat buffered from the clock kickback of the comparator) and the second creates OutP (Q) which is very well buffered from the comparator kickback.

The first inverter was also placed on OutN to act as a dummy load on that output.